[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anniepeacock/DANSAR/blob/devel_utils/notebooks/uavsar2tiff.ipynb)

1- Clone DANSAR Repo

In [ ]:
%cd /content
!git clone -b devel_utils https://github.com/anniepeacock/DANSAR.git

2- Install dependencies and get all DANSAR updates

In [ ]:
%cd /content/DANSAR
!git pull
!pip install -e .
!pip install asf-search requests numpy rasterio
!apt-get install -y -qq gdal-bin

3- User inputs

In [ ]:
from pathlib import Path
PRODUCT_NAME = "PanCan_20012_26007_004_260505_L090_CX_01"
OUTPUT_DIR = Path("/content/pancan_output")
BASE_URL = "https://uavsar.asf.alaska.edu/"

4- Import packages. If you see an error with DANSAR module, you might need to delete the DANSAR folder and clone it again

In [ ]:
import sys
%cd /content/DANSAR
sys.path.insert(0, "/content/DANSAR/src")
from urllib.parse import urlparse, parse_qs
from dansar.uavsar.infer_uavsar_paths import infer_uavsar_paths
from dansar.uavsar.download_uavsar import download_uavsar
from dansar.uavsar.ann2envi_header import ann2envi_header
from dansar.uavsar.make_tiff import make_tiff
from dansar.utils.make_asf_session import make_asf_session
import numpy as np
import subprocess
import rasterio
import xml.etree.ElementTree as ET

5- Download GRDs from Alaska Satellite Facility

In [ ]:
PRODUCT_NAME = PRODUCT_NAME.strip()
DOWNLOAD_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

paths = infer_uavsar_paths(
    base_url=BASE_URL,
    product_name=PRODUCT_NAME,
)

session = make_asf_session()

download_uavsar(
    urls=[paths["annotation_url"]],
    output_dir=DOWNLOAD_DIR,
    session=session,
    overwrite=False,
)

download_uavsar(
    urls=[paths["grd_hhhh_url"]],
    output_dir=DOWNLOAD_DIR,
    session=session,
    overwrite=False,
)

download_uavsar(
    urls=[paths["grd_hvhv_url"]],
    output_dir=DOWNLOAD_DIR,
    session=session,
    overwrite=False,
)

download_uavsar(
    urls=[paths["grd_vvvv_url"]],
    output_dir=DOWNLOAD_DIR,
    session=session,
    overwrite=False,
)

6- Make TIFFs for all GRDs in a folder

In [ ]:

OUTPUT_DIR = Path(OUTPUT_DIR)
PRODUCT_NAME = str(PRODUCT_NAME).strip()

OVERWRITE = False
NODATA = np.nan

PRODUCT_PREFIX = PRODUCT_NAME.replace("_CX_01", "")

ANN_PATH = OUTPUT_DIR / f"{PRODUCT_NAME}.ann"

if not ANN_PATH.exists():
    raise FileNotFoundError(f"Missing annotation: {ANN_PATH}")

HHHH_FILES = sorted(OUTPUT_DIR.glob(f"{PRODUCT_PREFIX}HHHH_*.grd"))
HVHV_FILES = sorted(OUTPUT_DIR.glob(f"{PRODUCT_PREFIX}HVHV_*.grd"))
VVVV_FILES = sorted(OUTPUT_DIR.glob(f"{PRODUCT_PREFIX}VVVV_*.grd"))

if len(HHHH_FILES) != 1:
    raise FileNotFoundError(f"Expected exactly one HHHH GRD, found {len(HHHH_FILES)}: {HHHH_FILES}")

if len(HVHV_FILES) != 1:
    raise FileNotFoundError(f"Expected exactly one HVHV GRD, found {len(HVHV_FILES)}: {HVHV_FILES}")

if len(VVVV_FILES) != 1:
    raise FileNotFoundError(f"Expected exactly one VVVV GRD, found {len(VVVV_FILES)}: {VVVV_FILES}")

grd_files = [
    HHHH_FILES[0],
    HVHV_FILES[0],
    VVVV_FILES[0],
]

# Make missing HDRs
for grd_path in grd_files:
    hdr_path = grd_path.with_name(f"{grd_path.name}.hdr")

    if hdr_path.exists() and not OVERWRITE:
        print("HDR exists:", hdr_path.name)
    else:
        print("Making HDR:", hdr_path.name)
        ann2envi_header(
            ann_path=ANN_PATH,
            group="grd_pwr",
            output_hdr=hdr_path,
        )

# Make COG TIFFs
for grd_path in grd_files:
    hdr_path = grd_path.with_name(f"{grd_path.name}.hdr")
    tif_path = grd_path.with_name(f"{grd_path.stem}_cog.tif")

    if tif_path.exists() and not OVERWRITE:
        print("TIFF exists:", tif_path.name)
    else:
        print("Making TIFF:", tif_path.name)

        make_tiff(
            raster_path=grd_path,
            header_path=hdr_path,
            output_dir=OUTPUT_DIR,
            nodata=NODATA,
            invalid_min=0.0,
            overwrite=OVERWRITE,
            suffix="_cog",
            extension=".tif",
        )

7- Make VRTs for all files in the user-specified product

In [ ]:
OUTPUT_DIR = Path(OUTPUT_DIR)
PRODUCT_NAME = str(PRODUCT_NAME).strip()

OVERWRITE = False

PRODUCT_PREFIX = PRODUCT_NAME.replace("_CX_01", "")

hhhh_cogs = sorted(OUTPUT_DIR.glob(f"{PRODUCT_PREFIX}HHHH_*_cog.tif"))
hvhv_cogs = sorted(OUTPUT_DIR.glob(f"{PRODUCT_PREFIX}HVHV_*_cog.tif"))
vvvv_cogs = sorted(OUTPUT_DIR.glob(f"{PRODUCT_PREFIX}VVVV_*_cog.tif"))

if len(hhhh_cogs) != 1:
    raise FileNotFoundError(f"Expected exactly one HHHH COG, found {len(hhhh_cogs)}: {hhhh_cogs}")

if len(hvhv_cogs) != 1:
    raise FileNotFoundError(f"Expected exactly one HVHV COG, found {len(hvhv_cogs)}: {hvhv_cogs}")

if len(vvvv_cogs) != 1:
    raise FileNotFoundError(f"Expected exactly one VVVV COG, found {len(vvvv_cogs)}: {vvvv_cogs}")

hhhh_tif = hhhh_cogs[0]
hvhv_tif = hvhv_cogs[0]
vvvv_tif = vvvv_cogs[0]

vrt_path = OUTPUT_DIR / f"{PRODUCT_NAME}_grd_pwr_stack.vrt"

if vrt_path.exists() and OVERWRITE:
    vrt_path.unlink()

if vrt_path.exists() and not OVERWRITE:
    pass
else:
    subprocess.run(
        [
            "gdalbuildvrt",
            "-separate",
            str(vrt_path),
            str(hhhh_tif),
            str(hvhv_tif),
            str(vvvv_tif),
        ],
        check=True,
    )

vrt_path